# Agentic RL Training with OpenShift AI Workbench

This notebook demonstrates how to submit and monitor agentic reinforcement learning training jobs from an OpenShift AI Workbench.

## Overview

- **Training Method**: PPO (Proximal Policy Optimization)
- **Model**: TinyLlama-1.1B (customizable)
- **Architecture**: Sidecar pattern with reward model
- **Execution**: Training runs in GPU pods, not in this notebook

## Prerequisites

- OpenShift AI Data Science Project created
- GPU quota allocated to your namespace
- Container images built and pushed (student-agent, reward-model)
- Kubeflow Training Operator installed

## 1. Setup and Configuration

In [ ]:
# Install required dependencies
!pip install -q pyyaml requests kubeflow-training

In [ ]:
import os
import sys
import yaml
import time
import subprocess
from datetime import datetime
from pathlib import Path

# Add SDK to path
sdk_path = Path.cwd() / 'sdk'
if sdk_path.exists():
    sys.path.insert(0, str(sdk_path))
    print(f"✓ SDK path added: {sdk_path}")
else:
    print(f"⚠ SDK path not found: {sdk_path}")
    print("  Make sure you're running this notebook from the agentic-rl/ directory")

In [ ]:
# Configuration
NAMESPACE = os.getenv('NAMESPACE', 'default')  # Your Data Science Project namespace
STUDENT_IMAGE = "quay.io/your-username/student-agent:latest"  # Update with your registry
REWARD_IMAGE = "quay.io/your-username/reward-model:latest"    # Update with your registry

# Machine Pool Configuration
# Choose which machine pool to target for training workloads
WORKLOAD_TYPE = "gpu-power"  # Options: "cpu-power", "gpu-power", "workers"

# GPU-Power Pool Configuration (for GPU training)
GPU_POOL_CONFIG = {
    "tolerations": [
        {"key": "nvidia.com/gpu", "operator": "Equal", "value": "present", "effect": "NoSchedule"},
        {"key": "workload", "operator": "Equal", "value": "gpu", "effect": "NoSchedule"}
    ],
    "nodeSelector": {
        "workload": "gpu-training",
        "nvidia.com/gpu.present": "true"
    }
}

# CPU-Power Pool Configuration (for CPU-intensive training)
CPU_POOL_CONFIG = {
    "tolerations": [
        {"key": "workload", "operator": "Equal", "value": "cpu", "effect": "NoSchedule"}
    ],
    "nodeSelector": {
        "workload": "cpu-intensive"
    }
}

# Workers Pool Configuration (general workloads, no taints)
WORKERS_POOL_CONFIG = {
    "tolerations": [],
    "nodeSelector": {
        "workload": "general"
    }
}

# Select configuration based on workload type
POOL_CONFIGS = {
    "gpu-power": GPU_POOL_CONFIG,
    "cpu-power": CPU_POOL_CONFIG,
    "workers": WORKERS_POOL_CONFIG
}

SELECTED_POOL_CONFIG = POOL_CONFIGS[WORKLOAD_TYPE]

print("Configuration:")
print(f"  Namespace: {NAMESPACE}")
print(f"  Student Image: {STUDENT_IMAGE}")
print(f"  Reward Image: {REWARD_IMAGE}")
print(f"\nMachine Pool Configuration:")
print(f"  Target Pool: {WORKLOAD_TYPE}")
print(f"  Tolerations: {SELECTED_POOL_CONFIG['tolerations']}")
print(f"  Node Selector: {SELECTED_POOL_CONFIG['nodeSelector']}")
print("")
print("⚠ Make sure to update the image URLs above with your container registry!")
print("⚠ Change WORKLOAD_TYPE to target different machine pools")

## 2. Initialize SDK

Import and initialize the Agentic RL Training SDK.

In [ ]:
from trainjob_sdk import AgenticRLTrainingSDK

# Initialize SDK
sdk = AgenticRLTrainingSDK(
    namespace=NAMESPACE,
    student_image=STUDENT_IMAGE,
    reward_model_image=REWARD_IMAGE,
)
print("✓ SDK initialized successfully")

## 3. Create TrainingRuntime

The TrainingRuntime defines the pod template with both student and reward model containers.

In [ ]:
# Create TrainingRuntime specification
print("Creating TrainingRuntime specification...")
runtime_spec = sdk.create_training_runtime(
    runtime_name="agentic-rl-pytorch",
    cluster_scoped=False  # Namespace-scoped
)

# Deploy TrainingRuntime
print("Deploying TrainingRuntime...")
sdk.deploy_runtime(runtime_spec, dry_run=False)
print("✓ TrainingRuntime deployed successfully")

In [ ]:
# Verify TrainingRuntime was created
!oc get trainingruntime -n {NAMESPACE}

## 4. Submit Training Job

Now let's submit an agentic RL training job with customizable parameters and GPU node tolerations.

In [ ]:
# Training configuration
JOB_NAME = f"agentic-rl-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
NUM_EPOCHS = 3
BATCH_SIZE = 4
LEARNING_RATE = 1e-5
PPO_EPOCHS = 4

print(f"Job Name: {JOB_NAME}")
print(f"Model: {MODEL_NAME}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"PPO Epochs: {PPO_EPOCHS}")

In [ ]:
# Submit training job via SDK with machine pool targeting (pure Pythonic approach)
print(f"Submitting training job via SDK to '{WORKLOAD_TYPE}' machine pool...")

# Create the job with tolerations and node selector built-in
job_id = sdk.create_train_job(
    name=JOB_NAME,
    model_name=MODEL_NAME,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    ppo_epochs=PPO_EPOCHS,
    num_nodes=1,
    gpu_per_node=2 if WORKLOAD_TYPE == "gpu-power" else 0,  # GPUs only for gpu-power pool
    memory_per_node="12Gi",
    cpu_per_node=6,
    # Pass tolerations and node selector directly to SDK
    tolerations=SELECTED_POOL_CONFIG['tolerations'],
    node_selector=SELECTED_POOL_CONFIG['nodeSelector'],
)

print(f"\n✓ Training job '{job_id}' submitted successfully!")
print(f"  ✓ Target machine pool: {WORKLOAD_TYPE}")
print(f"  ✓ Node selector: {SELECTED_POOL_CONFIG['nodeSelector']}")
print(f"  ✓ Tolerations: {len(SELECTED_POOL_CONFIG['tolerations'])} configured")

## 5. Monitor Training Job

In [ ]:
# Wait for pod to be scheduled and verify machine pool
import time

if job_id:
    print(f"Waiting for pod to be scheduled on '{WORKLOAD_TYPE}' machine pool...")
    for i in range(30):  # Wait up to 30 seconds
        result = subprocess.run(
            ["oc", "get", "pods", "-n", NAMESPACE, "-l", f"trainjob-name={job_id}",
             "-o", "jsonpath={.items[0].metadata.name}"],
            capture_output=True,
            text=True
        )
        
        if result.stdout.strip():
            pod_name = result.stdout.strip()
            print(f"✓ Pod created: {pod_name}")
            break
        time.sleep(1)
    else:
        print("⚠ Pod not found yet, may still be creating...")
        pod_name = None
    
    # Check which node the pod is scheduled on
    if pod_name:
        result = subprocess.run(
            ["oc", "get", "pod", pod_name, "-n", NAMESPACE,
             "-o", "jsonpath={.spec.nodeName}"],
            capture_output=True,
            text=True
        )
        
        node_name = result.stdout.strip()
        if node_name:
            print(f"✓ Pod scheduled on node: {node_name}")
            
            # Get node labels to verify machine pool
            result = subprocess.run(
                ["oc", "get", "node", node_name, "-o", "json"],
                capture_output=True,
                text=True
            )
            
            if result.returncode == 0:
                import json
                node_info = json.loads(result.stdout)
                labels = node_info.get('metadata', {}).get('labels', {})
                
                # Check workload label
                workload_label = labels.get('workload', 'unknown')
                print(f"  Node workload label: {workload_label}")
                
                # Verify it matches expected pool
                expected_workload = SELECTED_POOL_CONFIG['nodeSelector'].get('workload', 'unknown')
                if workload_label == expected_workload:
                    print(f"✓ Correct machine pool! Pod is on '{WORKLOAD_TYPE}' pool")
                else:
                    print(f"⚠ Warning: Expected workload='{expected_workload}', but node has '{workload_label}'")
                
                # Check GPU labels if using gpu-power pool
                if WORKLOAD_TYPE == "gpu-power":
                    if "nvidia.com/gpu.present" in labels:
                        print("✓ Node has GPU labels")
                    else:
                        print("⚠ Node missing GPU labels")
                
                # Show instance type
                instance_type = labels.get('node.kubernetes.io/instance-type', 'unknown')
                print(f"  Instance type: {instance_type}")
                
            # Check tolerations on the pod
            result = subprocess.run(
                ["oc", "get", "pod", pod_name, "-n", NAMESPACE,
                 "-o", "jsonpath={.spec.tolerations}"],
                capture_output=True,
                text=True
            )
            
            if result.returncode == 0:
                num_expected_tolerations = len(SELECTED_POOL_CONFIG['tolerations'])
                tolerations_str = result.stdout
                print(f"✓ Pod has tolerations configured ({num_expected_tolerations} expected)")
                
                # Verify specific tolerations
                for tol in SELECTED_POOL_CONFIG['tolerations']:
                    if tol['key'] in tolerations_str:
                        print(f"  ✓ Toleration '{tol['key']}' present")
                    else:
                        print(f"  ⚠ Toleration '{tol['key']}' missing")
        else:
            print("⚠ Pod not scheduled yet")
else:
    print("⚠ Job ID not set, skipping verification")

## 4a. Verify GPU Node Scheduling

If your GPU nodes have taints (e.g., `nvidia.com/gpu=present:NoSchedule`), verify that the pod was scheduled correctly on a GPU node.

In [ ]:
# Check job status
!oc get trainjob {job_id} -n {NAMESPACE} -o wide

In [ ]:
# Check associated pods
!oc get pods -n {NAMESPACE} -l trainjob-name={job_id}

In [ ]:
# Get detailed job information
!oc describe trainjob {job_id} -n {NAMESPACE}

## 6. View Training Logs

Monitor the training progress in real-time.

In [ ]:
# View student training logs (last 50 lines)
!oc logs -l trainjob-name={job_id} -c node -n {NAMESPACE} --tail=50

In [ ]:
# View reward model logs
!oc logs -l trainjob-name={job_id} -c reward-model -n {NAMESPACE} --tail=20

In [ ]:
# Follow logs in real-time (run this cell and stop it manually when done)
# Warning: This will run continuously until interrupted
!oc logs -f -l trainjob-name={job_id} -c node -n {NAMESPACE}

## 7. Monitor Training Progress

Parse logs to extract training metrics.

In [ ]:
import re

def get_training_metrics(job_name, namespace):
    """Extract training metrics from logs."""
    result = subprocess.run(
        ["oc", "logs", "-l", f"trainjob-name={job_name}", "-c", "node", "-n", namespace],
        capture_output=True,
        text=True
    )
    
    metrics = []
    for line in result.stdout.split('\n'):
        if 'Avg Reward' in line:
            # Parse metrics from log line
            match = re.search(r'Avg Reward: ([0-9.]+)', line)
            if match:
                metrics.append(float(match.group(1)))
    
    return metrics

# Get current metrics
metrics = get_training_metrics(job_id, NAMESPACE)
if metrics:
    print(f"Found {len(metrics)} training steps")
    print(f"Latest average reward: {metrics[-1]:.4f}")
    print(f"Best reward so far: {max(metrics):.4f}")
else:
    print("No metrics found yet. Training may still be starting up.")

In [ ]:
# Plot training progress (if metrics available)
if metrics:
    import matplotlib.pyplot as plt
    
    plt.figure(figsize=(10, 5))
    plt.plot(metrics, marker='o')
    plt.xlabel('Training Step')
    plt.ylabel('Average Reward')
    plt.title('Agentic RL Training Progress')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"\nSummary Statistics:")
    print(f"  Steps: {len(metrics)}")
    print(f"  Mean Reward: {sum(metrics)/len(metrics):.4f}")
    print(f"  Min Reward: {min(metrics):.4f}")
    print(f"  Max Reward: {max(metrics):.4f}")

## 8. Access Training Checkpoints

Once training completes, copy checkpoints to your notebook storage.

In [ ]:
# Get pod name
result = subprocess.run(
    ["oc", "get", "pods", "-n", NAMESPACE, "-l", f"trainjob-name={job_id}", 
     "-o", "jsonpath={.items[0].metadata.name}"],
    capture_output=True,
    text=True
)

pod_name = result.stdout.strip()
if pod_name:
    print(f"Pod name: {pod_name}")
else:
    print("No pod found for this job")

In [ ]:
# List available checkpoints in the pod
if pod_name:
    !oc exec {pod_name} -c node -n {NAMESPACE} -- ls -lh /checkpoints

In [ ]:
# Copy checkpoints to notebook storage
if pod_name:
    checkpoint_dir = f"./checkpoints/{job_id}"
    !mkdir -p {checkpoint_dir}
    !oc cp {pod_name}:/checkpoints/final {checkpoint_dir}/final -c node -n {NAMESPACE}
    print(f"✓ Checkpoints copied to {checkpoint_dir}/final")

## 9. Test the Fine-Tuned Model

Load and test the fine-tuned model locally in the notebook.

In [ ]:
# Install transformers if not already installed
!pip install -q transformers torch

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load the fine-tuned model
checkpoint_path = f"./checkpoints/{job_id}/final"

if Path(checkpoint_path).exists():
    print(f"Loading model from {checkpoint_path}...")
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
    model = AutoModelForCausalLM.from_pretrained(
        checkpoint_path,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None
    )
    print("✓ Model loaded successfully")
else:
    print(f"✗ Checkpoint not found at {checkpoint_path}")
    print("  Make sure training has completed and checkpoints were copied.")

In [ ]:
# Test the model with sample prompts
test_prompts = [
    "How do I make chocolate chip cookies?",
    "Explain the water cycle to a 10-year-old.",
    "Write a short poem about the ocean.",
]

for prompt in test_prompts:
    print(f"\n{'='*80}")
    print(f"Prompt: {prompt}")
    print(f"{'-'*80}")
    
    inputs = tokenizer(prompt, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=150,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Remove the prompt from response
    response = response[len(prompt):].strip()
    print(f"Response: {response}")

## 10. Run Hyperparameter Sweep

Submit multiple jobs with different hyperparameters for experimentation.

In [ ]:
import pandas as pd

# Define experiments
experiments = [
    {"learning_rate": 1e-5, "batch_size": 4, "ppo_epochs": 4},
    {"learning_rate": 2e-5, "batch_size": 4, "ppo_epochs": 4},
    {"learning_rate": 1e-5, "batch_size": 8, "ppo_epochs": 4},
    {"learning_rate": 1e-5, "batch_size": 4, "ppo_epochs": 8},
]

# Submit jobs (uncomment to run)
# experiment_jobs = []
# 
# for i, config in enumerate(experiments):
#     exp_name = f"rl-exp-{i}-lr{config['learning_rate']}-bs{config['batch_size']}"
#     
#     # Submit using SDK
#     exp_job_id = sdk.create_train_job(
#         name=exp_name,
#         model_name=MODEL_NAME,
#         num_epochs=NUM_EPOCHS,
#         **config
#     )
#     
#     experiment_jobs.append({**config, 'job_id': exp_job_id, 'status': 'submitted'})
#     print(f"✓ Submitted experiment {i+1}/{len(experiments)}: {exp_job_id}")
# 
# # Display experiment tracking table
# df = pd.DataFrame(experiment_jobs)
# display(df)

print("Hyperparameter sweep code is commented out.")
print("Uncomment the code above to run multiple experiments in parallel.")

## 11. Clean Up Resources

In [ ]:
# Delete the training job
!oc delete trainjob {job_id} -n {NAMESPACE} --ignore-not-found
print(f"✓ Deleted TrainJob: {job_id}")

In [ ]:
# Delete the training runtime (if you want to clean up completely)
# Uncomment to run:
# !oc delete trainingruntime agentic-rl-pytorch -n {NAMESPACE} --ignore-not-found
# print("✓ Deleted TrainingRuntime")

print("TrainingRuntime deletion is commented out.")
print("Uncomment to delete the runtime (you'll need to recreate it for future jobs).")

## 12. Utility Functions

In [ ]:
def list_all_jobs(namespace):
    """List all training jobs in the namespace."""
    result = subprocess.run(
        ["oc", "get", "trainjob", "-n", namespace, "-o", "wide"],
        capture_output=True,
        text=True
    )
    print(result.stdout)

def get_job_status(job_name, namespace):
    """Get status of a specific job."""
    result = subprocess.run(
        ["oc", "get", "trainjob", job_name, "-n", namespace, "-o", "jsonpath={.status.phase}"],
        capture_output=True,
        text=True
    )
    return result.stdout.strip() if result.returncode == 0 else "Unknown"

def wait_for_job_completion(job_name, namespace, timeout=3600, check_interval=30):
    """Wait for a job to complete."""
    print(f"Waiting for job '{job_name}' to complete...")
    start_time = time.time()
    
    while time.time() - start_time < timeout:
        status = get_job_status(job_name, namespace)
        print(f"  Status: {status}")
        
        if status in ['Succeeded', 'Failed']:
            print(f"\n✓ Job completed with status: {status}")
            return status
        
        time.sleep(check_interval)
    
    print(f"\n✗ Timeout waiting for job completion")
    return "Timeout"

print("✓ Utility functions loaded")

In [ ]:
# Example: List all jobs
list_all_jobs(NAMESPACE)

In [ ]:
# Example: Wait for current job to complete
# final_status = wait_for_job_completion(job_id, NAMESPACE, timeout=3600, check_interval=30)

## Summary

This notebook demonstrated:

1. ✅ Setting up the SDK and configuration
2. ✅ Creating a TrainingRuntime with sidecar architecture
3. ✅ Submitting agentic RL training jobs
4. ✅ Monitoring job status and logs
5. ✅ Visualizing training metrics
6. ✅ Accessing and testing trained models
7. ✅ Running hyperparameter sweeps
8. ✅ Cleaning up resources

## Next Steps

- Customize training prompts in `student/environment.py`
- Experiment with different models (Phi-2, Mistral-7B)
- Add persistent storage for checkpoints
- Integrate Weights & Biases for experiment tracking
- Scale to multi-node distributed training

## Resources

- [README.md](README.md) - Full documentation
- [SDK Documentation](sdk/README.md)
- [Kubeflow Training Operator](https://www.kubeflow.org/docs/components/trainer/)
- [OpenShift AI Documentation](https://access.redhat.com/documentation/en-us/red_hat_openshift_ai_self-managed/)